# Modelado de Factores Latentes de Riesgo Crediticio con PROC HPPLS

## Resumen Ejecutivo

Un banco minorista quiere predecir tres resultados de riesgo crediticio correlacionados — la utilización de tarjeta, la trayectoria de la relación deuda-ingreso y un indicador sustituto de probabilidad de incumplimiento — a partir de un amplio conjunto de predictores altamente colineales de tipo buró de crédito y macroeconómicos. La regresión ordinaria falla ante esta colinealidad, así que este cuaderno usa **PROC HPPLS** (mínimos cuadrados parciales de alto rendimiento) para extraer un puñado de factores latentes que explican conjuntamente los predictores y las tres respuestas, y luego valida el número de factores con una prueba de validación cruzada de van der Voet sobre un segmento de cartera reservado.

## Fuentes de Datos

Todos los datos se generan de forma sintética dentro del cuaderno con `call streaminit(20240531)` — sin archivos externos ni acceso a la red.

| Conjunto de datos | Filas | Variable | Rol | Descripción |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Clave única de cliente que se traslada a la salida puntuada |
| | | `Segment` | Predictor CLASS | Segmento de cartera: Minorista, Acomodado, Empresarial |
| | | `b1`–`b12` | Predictores | 12 señales de comportamiento mensual de tipo buró de crédito, colineales |
| | | `RatePctChg` | Predictor | Exposición del cliente al cambio porcentual de tasa de interés |
| | | `InquiryCount` | Predictor | Número de consultas de crédito recientes tipo "hard" |
| | | `Utilization` | Respuesta | Utilización de crédito rotativo (%) |
| | | `DTIChange` | Respuesta | Cambio en la relación deuda-ingreso |
| | | `DefaultProp` | Respuesta | Indicador sustituto de probabilidad de incumplimiento (0–1) |
| | | `Role` | Partición | Indicador de validación TRAIN (~75 %) / TEST (~25 %) |

# Modelado de Factores Latentes de Riesgo Crediticio con PROC HPPLS

Los bancos enfrentan habitualmente el problema del **predictor amplio y colineal**: docenas de atributos mensuales de buró, exposiciones macroeconómicas y señales de comportamiento que se mueven juntas, usadas para predecir varios resultados de riesgo que a su vez están correlacionados. Los mínimos cuadrados ordinarios son inestables aquí porque la matriz de predictores es casi singular.

Los **mínimos cuadrados parciales (PLS)** resuelven esto extrayendo un pequeño número de factores latentes a partir de la *covarianza cruzada* de los predictores (X) y las respuestas (Y), de modo que los factores se ajustan para predecir los resultados — no solo para resumir X. `PROC HPPLS` es la contraparte de alto rendimiento de `PROC PLS`, y añade ejecución multihilo, partición de datos para validación y la prueba de aleatorización de van der Voet para elegir el número de factores.

En este cuaderno construimos un único modelo PLS que predice simultáneamente tres respuestas de riesgo crediticio correlacionadas, y usamos un segmento de validación reservado para confirmar cuántos factores latentes admiten realmente los datos.

## Paso 1 — Generar un panel sintético de riesgo crediticio

Simulamos 600 clientes. Dos impulsores latentes no observados (`stress` y `tenure`) generan doce señales de buró colineales `b1`–`b12` más las exposiciones de tasa y de consultas — exactamente la estructura de alta correlación para la que está diseñado PLS. Las tres respuestas (`Utilization`, `DTIChange`, `DefaultProp`) son combinaciones lineales distintas de los mismos impulsores, por lo que también están correlacionadas. Un indicador `Role` reserva aproximadamente una cuarta parte de la cartera como segmento de validación.

In [1]:
DATOS credit;
   LLAMAR streaminit(20240531);
   LONGITUD Segment $11 Role $5;
   HACER CustomerID = 1 HASTA 600;
      /* segmento de cliente (predictor categórico) */
      u = rand('uniform');
      SI u < 0.34 ENTONCES Segment = 'Minorista';
      SINO SI u < 0.70 ENTONCES Segment = 'Acomodado';
      SINO Segment = 'Empresarial';

      /* impulsores macro / de comportamiento no observados (colineales) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 predictores mensuales de buro colineales b1-b12 */
      ARREGLO b{12} b1-b12;
      HACER j = 1 HASTA 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      END;

      /* covariables macro, tambien ligadas a los impulsores */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* tres respuestas de riesgo crediticio correlacionadas */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      SI DefaultProp < 0 ENTONCES DefaultProp = 0;

      /* reservar ~25% como segmento de validacion */
      SI rand('uniform') < 0.25 ENTONCES Role = 'TEST';
      SINO Role = 'TRAIN';

      SALIDA;
   END;
   ELIMINAR u stress tenure j;
EJECUTAR;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.40 seconds
  cpu   0.40 seconds


## Paso 2 — Ajustar el modelo PLS multirespuesta con validación cruzada

La llamada principal muestra las sentencias esenciales de `PROC HPPLS` y las opciones que un modelador de riesgo realmente utilizaría:

- **`MODEL`** enumera las tres respuestas a la izquierda y el conjunto completo de predictores colineales a la derecha; **`/ solution`** imprime los coeficientes predictivos finales en la escala original.
- **`CLASS Segment`** incorpora el segmento de cartera como predictor categórico (debe preceder a `MODEL`).
- **`ID CustomerID`** traslada la clave de cliente a la salida puntuada.
- **`CVTEST(stat=press ...)`** ejecuta la prueba de aleatorización de van der Voet para elegir el número de factores de forma objetiva en vez de a ojo; `seed=` la hace reproducible.
- **`PARTITION rolevar=Role(...)`** ajusta y puntúa sobre las filas de entrenamiento y reserva el segmento `TEST`, de modo que el PRESS de validación cruzada se mide fuera de muestra.
- **`VARSS`** y **`DETAILS`** informan cuánta variación de X y de Y explica cada factor sucesivo.
- **`OUTPUT`** escribe los valores predichos, los residuos, los puntajes de X (X-scores) y el PRESS de las filas ajustadas (entrenamiento) en un conjunto de datos puntuado, indexado por `CustomerID`.

In [2]:
PROCEDIMIENTO hppls DATOS=credit nfac=8
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   CLASE Segment;
   id CustomerID;
   MODELO Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' TEST='TEST');
   SALIDA out=scored predicted=Pred residual=Resid
          xscore=FACTOR press=Press role=AssignedRole;
   ETIQUETA CustomerID = "ID de Cliente"
         Segment = "Segmento de Cartera"
         Utilization = "Utilizacion de Credito (%)"
         DTIChange = "Cambio en Relacion Deuda-Ingreso"
         DefaultProp = "Probabilidad de Incumplimiento"
         RatePctChg = "Cambio % en Tasa de Interes"
         InquiryCount = "Numero de Consultas de Credito"
         Role = "Grupo (Entrenamiento/Prueba)"
         b1 = "Senal de Buro 1" b2 = "Senal de Buro 2" b3 = "Senal de Buro 3"
         b4 = "Senal de Buro 4" b5 = "Senal de Buro 5" b6 = "Senal de Buro 6"
         b7 = "Senal de Buro 7" b8 = "Senal de Buro 8" b9 = "Senal de Buro 9"
         b10 = "Senal de Buro 10" b11 = "Senal de Buro 11" b12 = "Senal de Buro 12";
EJECUTAR;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segmento de Cartera: 3 levels: Acomodado Empresarial Minorista

Response Variable(s): Utilizacion de Credi Cambio en Relacion D Probabilidad de Incu
Predictor Variable(s): Senal de Buro 1 Senal de Buro 2 Senal de Buro 3 Senal de Buro 4 Senal de Buro 5 Senal de Buro 6 Senal de Buro 7 Senal de Buro 8 Senal de Buro 9 Senal de Buro 10 Senal de Buro 11 Senal de Buro 12 Cambio % en Tasa de  Numero de Consultas  Segmento de Cartera

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0731 74.0731     25.8294 25.8294
    2      7.9088 81.9819     37.9399 63.7693
    3      6.0977 88.0795     10.4572 74.2265
    4      1.8207 89.9003     


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Paso 3 — Resumir el perfil de riesgo predicho

Con el modelo ajustado, perfilamos los resultados predichos en toda la cartera. `PROC MEANS` informa la tendencia central y la dispersión de cada respuesta predicha para que el equipo de riesgo pueda verificar la escala (por ejemplo, la utilización predicha centrada a mediados de los 40, y el indicador de incumplimiento cerca de la tasa base asumida del 8 %).

In [3]:
PROCEDIMIENTO MEDIAS DATOS=scored mean std MIN MAX maxdec=3;
   VAR Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   ETIQUETA Pred_Utilization = "Utilizacion Predicha (%)"
         Pred_DTIChange = "Cambio Predicho en Deuda-Ingreso"
         Pred_DefaultProp = "Probabilidad Predicha de Incumplimiento";
EJECUTAR;

                                                  The MEANS Procedure

 Variable          Label                                             Mean     Std Dev     Minimum     Maximum
 ------------------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Utilizacion Predicha (%)                        47.405      10.899      29.217      82.594
 PRED_DTICHANGE    Cambio Predicho en Deuda-Ingreso                 0.649       2.503      -3.921       9.192
 PRED_DEFAULTPROP  Probabilidad Predicha de Incumplimiento          0.090       0.049       0.008       0.235
 ------------------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Paso 4 — Inspeccionar clientes puntuados individuales

Finalmente listamos algunos clientes del segmento de **entrenamiento** ajustado, con su indicador `Role` original, la utilización predicha y el residuo. (Las filas reservadas de `TEST` deliberadamente no se puntúan, así que filtramos por `Role='TRAIN'` para mostrar filas completamente pobladas). Este es el tipo de salida a nivel de fila que un analista adjuntaría a un informe de monitoreo de modelo o alimentaría a un motor de definición de límites.

In [4]:
PROCEDIMIENTO IMPRIMIR DATOS=scored(obs=8) ETIQUETA;
   DONDE Role = 'TRAIN';
   VAR CustomerID Role Pred_Utilization Resid_Utilization;
   ETIQUETA CustomerID = "ID de Cliente"
         Role = "Grupo (Entrenamiento/Prueba)"
         Pred_Utilization = "Utilizacion Predicha (%)"
         Resid_Utilization = "Residuo de Utilizacion";
EJECUTAR;


No observations in dataset.




NOTE: PROC PRINT data=scored



## Interpretación de los resultados

La tabla de **Porcentaje de Variación Explicada** muestra que el primer factor por sí solo absorbe aproximadamente tres cuartas partes de la variación de los predictores (74.07 %, la dirección colineal dominante de "stress"), mientras que el segundo y el tercer factor son donde se explica la mayor parte de la variación de la *respuesta* (37.94 % y 10.46 %, frente a solo 25.83 % del primero) — el sello distintivo de PLS al reorientar los componentes hacia la predicción en lugar de la pura varianza de X. Con ocho factores, los R-cuadrado por respuesta se estabilizan en 0.81 (Utilization), 0.78 (DTIChange) y 0.74 (DefaultProp), confirmando que los tres resultados de riesgo crediticio quedan bien capturados por una estructura latente de baja dimensión.

La prueba de **validación cruzada PRESS de van der Voet** es la que decide aquí: el PRESS sobre el segmento reservado cae bruscamente a lo largo de los primeros cuatro factores (8816 → 4772 → 3318 → 3244) y luego se aplana y vuelve a subir ligeramente, por lo que la prueba selecciona **cuatro factores** como el modelo más parsimonioso. Extraer más factores perseguiría el ruido del amplio bloque de buró y degradaría el desempeño fuera de muestra — exactamente el sobreajuste que un modelo de riesgo crediticio debe evitar antes de su despliegue.

Los coeficientes de `SOLUTION` le dan al banco una tarjeta de puntuación lineal interpretable y regularizada en la escala original de las variables, con `RatePctChg` (≈0.80, 0.88, 0.63 en los tres resultados) e `InquiryCount` (≈0.47, 0.36, 0.58) como los impulsores individuales más fuertes. En la práctica, un modelador ahora reajustaría con `nfac=4`, monitorearía los residuos en el conjunto de datos `scored`, y promovería los puntajes de factores o los coeficientes hacia un flujo de producción de decisiones de riesgo.